# Extracting q-Fano candidates from the saved classification

- We consider the 94 canonical representatives of rank-3 q-matroids on $\mathrm{GF}(2)^5$.
- We extract 10 candidates using the conditions on flats satisfied by the restriction of a q-Fano plane to a 5-dimensional subspace.

In [2]:
from collections import Counter
from pathlib import Path

from sage.all import GF, VectorSpace, table

from qmatroid.finite_geometry import (
    ReversedLexicographicOrderOfSubspaces,
    get_subspace_tables,
)
from qmatroid.q_matroid_enumeration import load_q_matroids

q = 2
n = 5
k = 3

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
path = project_root / "results" / "q2_dimension5_rank3.txt"
tables = get_subspace_tables(q, n)
Ms = load_q_matroids(path, q=q, n=n)
V = VectorSpace(GF(q), n)
order = ReversedLexicographicOrderOfSubspaces(V)
subspaces = tuple(order.sorted())
subspace_id = {X: i for i, X in enumerate(subspaces)}

display(table(
    [
        ("file", str(path.relative_to(project_root))),
        ("field order", q),
        ("ambient dimension", n),
        ("stored rank-3 representatives", len(Ms)),
    ],
    header_row=["item", "value"],
))

  item                            value
├───────────────────────────────┼─────────────────────────────────┤
  file                            results/q2_dimension5_rank3.txt
  field order                     2
  ambient dimension               5
  stored rank-3 representatives   94

## Conditions on rank-2 flats

- In the restriction of a q-Fano plane, the rank-2 flats have the following dimension distribution.
  - 2-dimensional subspaces: 120.
  - 3-dimensional subspaces: 5.
- We select the saved canonical representatives with this dimension distribution.
- We display the number of candidates and their isomorphism class numbers in the original classification results.

In [3]:
def flat_dimensions(M):
    return Counter(
        tables.subspace_dimensions[F]
        for F in M.flat_ids(tables, rank=2)
    )

required = Counter({2: 120, 3: 5})
candidates = tuple(
    (i, M)
    for i, M in enumerate(Ms, start=1)
    if flat_dimensions(M) == required
)

display(table(
    [
        ("rank-3 q-matroids examined", len(Ms)),
        ("candidates retained", len(candidates)),
        ("rank-3 classes", [i for i, M in candidates]),
    ],
    header_row=["item", "value"],
))

  item                         value
├────────────────────────────┼─────────────────────────────────────────┤
  rank-3 q-matroids examined   94
  candidates retained          10
  rank-3 classes               [1, 20, 21, 22, 28, 29, 55, 57, 60, 77]

## Candidates and their linear automorphism groups

- For each candidate, we list the following values.
  - The candidate number and its isomorphism class number in the original classification results.
  - Whether it is a canonical representative.
  - The numbers of 2-dimensional and 3-dimensional rank-2 flats.
  - The order and structure of its linear automorphism group.

In [4]:
groups = tuple(
    M.linear_automorphism_group(tables)
    for i, M in candidates
)
print(
    f"Selected {len({M.rank_table for i, M in candidates})} "
    "canonical rank tables."
)
display(table(
    [
        (
            number,
            i,
            M.is_canonical(tables),
            flat_dimensions(M).get(2, 0),
            flat_dimensions(M).get(3, 0),
            G.order(),
            G.structure_description(),
        )
        for number, ((i, M), G) in enumerate(
            zip(candidates, groups), start=1
        )
    ],
    header_row=[
        "candidate",
        "rank-3 class",
        "canonical",
        "2D rank-2 flats",
        "3D rank-2 flats",
        "GL order",
        "GL structure",
    ],
))

Selected 10 canonical rank tables.


  candidate   rank-3 class   canonical   2D rank-2 flats   3D rank-2 flats   GL order   GL structure
├───────────┼──────────────┼───────────┼─────────────────┼─────────────────┼──────────┼─────────────────────────────────┤
  1           1              True        120               5                 5760       (C2 x C2 x C2 x C2) : (A5 : S3)
  2           20             True        120               5                 5          C5
  3           21             True        120               5                 8          D4
  4           22             True        120               5                 120        S5
  5           28             True        120               5                 3          C3
  6           29             True        120               5                 12         D6
  7           55             True        120               5                 2          C2
  8           57             True        120               5                 8          D4
  9           60    

## The five 3-dimensional hyperplanes

- For the five 3-dimensional hyperplanes of each candidate, we display the subspace IDs and representative matrices.
- The representative matrices use the reverse canonical representation, which defines the subspace order in the paper.
- The row space of each matrix is the corresponding hyperplane.

In [5]:
hyperplanes = []
for number, (i, M) in enumerate(candidates, start=1):
    Hs = tuple(
        F
        for F in M.hyperplane_ids(tables)
        if tables.subspace_dimensions[F] == 3
    )
    hyperplanes.append(Hs)
    print(f"Candidate {number} (rank-3 class {i})")
    display(table(
        [
            (
                i,
                H,
                str([
                    list(row)
                    for row in order.canonical_reversed_representation(subspaces[H]).rows()
                ]),
            )
            for i, H in enumerate(Hs, start=1)
        ],
        header_row=["hyperplane", "subspace ID", "matrix"],
    ))

Candidate 1 (rank-3 class 1)


  hyperplane   subspace ID   matrix
├────────────┼─────────────┼─────────────────────────────────────────────────────┤
  1            8             [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]]
  2            84            [[1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  3            220           [[1, 0, 0, 0, 0], [0, 0, 1, 1, 0], [0, 1, 0, 0, 1]]
  4            279           [[1, 0, 0, 0, 0], [0, 1, 1, 1, 0], [0, 0, 1, 0, 1]]
  5            306           [[1, 0, 0, 0, 0], [0, 1, 0, 1, 0], [0, 1, 1, 0, 1]]

Candidate 2 (rank-3 class 20)


  hyperplane   subspace ID   matrix
├────────────┼─────────────┼─────────────────────────────────────────────────────┤
  1            8             [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]]
  2            84            [[1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  3            112           [[0, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  4            170           [[0, 1, 1, 0, 0], [0, 1, 0, 1, 0], [1, 0, 0, 0, 1]]
  5            205           [[1, 1, 1, 0, 0], [1, 0, 0, 1, 0], [0, 1, 0, 0, 1]]

Candidate 3 (rank-3 class 21)


...

  hyperplane   subspace ID   matrix
├────────────┼─────────────┼─────────────────────────────────────────────────────┤
  1            8             [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]]
  2            84            [[1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  3            112           [[0, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  4            121           [[0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  5            220           [[1, 0, 0, 0, 0], [0, 0, 1, 1, 0], [0, 1, 0, 0, 1]]

Candidate 9 (rank-3 class 60)


  hyperplane   subspace ID   matrix
├────────────┼─────────────┼─────────────────────────────────────────────────────┤
  1            8             [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]]
  2            84            [[1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  3            112           [[0, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  4            121           [[0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  5            178           [[0, 1, 0, 0, 0], [0, 0, 1, 1, 0], [1, 0, 0, 0, 1]]

Candidate 10 (rank-3 class 77)


  hyperplane   subspace ID   matrix
├────────────┼─────────────┼─────────────────────────────────────────────────────┤
  1            8             [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]]
  2            84            [[1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  3            112           [[0, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  4            121           [[0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1]]
  5            127           [[1, 1, 0, 0, 0], [0, 0, 1, 1, 0], [0, 0, 0, 0, 1]]

## Images of hyperplanes under the generator matrices

- We display the generators of the linear automorphism group as matrices.
- Each matrix acts on the row spaces of the hyperplanes from the right.
- The last column lists the subspace IDs of the images of the five hyperplanes in the preceding table, in the same order.

In [6]:
for number, ((i, M), G, Hs) in enumerate(
    zip(candidates, groups, hyperplanes),
    start=1,
):
    print(f"Candidate {number}")
    display(table(
        [(G.order(), G.structure_description(), len(G.gens()))],
        header_row=["GL order", "GL structure", "number of generators"],
    ))
    display(table(
        [
            (
                i,
                str([list(row) for row in g.matrix().rows()]),
                tuple(subspace_id[subspaces[H] * g] for H in Hs),
            )
            for i, g in enumerate(G.gens(), start=1)
        ],
        header_row=[
            "generator",
            "matrix",
            "images of the five hyperplane IDs",
        ],
    ))

Candidate 1


  GL order   GL structure                      number of generators
├──────────┼─────────────────────────────────┼──────────────────────┤
  5760       (C2 x C2 x C2 x C2) : (A5 : S3)   8

  generator   matrix                                                                                  images of the five hyperplane IDs
├───────────┼───────────────────────────────────────────────────────────────────────────────────────┼───────────────────────────────────┤
  1           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [1, 0, 1, 1, 1], [1, 1, 0, 1, 0]]   (8, 306, 84, 279, 220)
  2           [[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 0, 0, 0, 1]]   (8, 84, 220, 279, 306)
  3           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 0, 0, 0, 1]]   (8, 84, 220, 279, 306)
  4           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 0, 0, 0, 1]]   (8, 84, 220, 279, 306)
  5           [[1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0], [1, 0, 0, 1, 0], [1, 0, 0, 0, 1]]   (8, 84, 220, 279, 306)
  6           [[1, 0, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 1, 1, 1], [1, 1, 0, 0, 0], [0, 1, 1, 0, 0]]  

Candidate 2


  GL order   GL structure   number of generators
├──────────┼──────────────┼──────────────────────┤
  5          C5             1

...

  GL order   GL structure   number of generators
├──────────┼──────────────┼──────────────────────┤
  2          C2             1

  generator   matrix                                                                                  images of the five hyperplane IDs
├───────────┼───────────────────────────────────────────────────────────────────────────────────────┼───────────────────────────────────┤
  1           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 1], [0, 0, 0, 0, 1]]   (8, 84, 121, 112, 178)

Candidate 10


  GL order   GL structure   number of generators
├──────────┼──────────────┼──────────────────────┤
  48         C2 x S4        4

  generator   matrix                                                                                  images of the five hyperplane IDs
├───────────┼───────────────────────────────────────────────────────────────────────────────────────┼───────────────────────────────────┤
  1           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 0], [0, 0, 0, 0, 1]]   (8, 84, 121, 112, 127)
  2           [[1, 0, 0, 0, 0], [1, 0, 1, 0, 0], [1, 1, 0, 0, 0], [1, 0, 0, 1, 0], [0, 0, 0, 0, 1]]   (8, 84, 127, 121, 112)
  3           [[0, 0, 1, 0, 0], [0, 1, 0, 0, 0], [1, 1, 0, 0, 0], [0, 1, 1, 1, 0], [0, 0, 0, 0, 1]]   (8, 112, 127, 84, 121)
  4           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 1], [0, 0, 0, 0, 1]]   (8, 84, 121, 112, 127)